# Model Inference Analysis — Shared XGBoost and GAM Workflow

This notebook loads one exported modelling run through its manifest and produces global and cohort-level interpretation artifacts for the final selected model.<br>
**Workflow summary:** load the run context and final-model artifacts, export unified per-row feature effects plus a global feature-effect ranking, inspect feature effects, compare cohorts by actual performance, then save the analysis plots/tables.


## 1. Imports and Configuration
**Purpose:** Load analysis libraries plus the manifest-aware helper functions needed to reconstruct a completed modelling run.<br>
**Inputs:** local `src/` package path, model/run identifiers, and plotting dependencies.<br>
**Outputs:** imported analysis libraries and notebook configuration constants.<br>
**How to Verify:** the import/configuration cells should print the requested run identifiers without import errors.


In [1]:
# Ensure the notebook can import local helper modules even when it is launched
# from a nested working directory inside the repository.
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "data_modelling").exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Could not locate repo src/ directory for notebook imports.")

In [2]:
# Core libraries
import warnings
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.inspection import PartialDependenceDisplay, partial_dependence

import xgboost as xgb
import shap

from data_modelling.prepared_data import MODEL_SETTING_COLS
from data_modelling.run_context import (
    format_exported_model_label,
    get_exported_model_info,
    load_run_context,
)
from data_modelling.feature_effect_performance_regimes_utils import (
    build_feature_effect_importance_table,
    compute_gam_feature_effects,
    prepare_feature_effect_export,
    resolve_raw_metric_col,
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

MODEL_ID = 'xgboost'
RUN_NAME = 'full_trainval_12ep_1seed_MI_correct'
TARGET_COL = None
LOWER_IS_BETTER = True  # Current interpretable targets are trajectory error metrics.

RESULTS_ROOT = Path("../../results/interpretable_model") / MODEL_ID / RUN_NAME
TABLES_DIR = RESULTS_ROOT / "tables"
PLOTS_DIR = RESULTS_ROOT / "plots"

print(f"Results root: {RESULTS_ROOT.resolve()}")
print(f"Model ID: {MODEL_ID}")
print(f"Run: {RUN_NAME}")
print(f"TARGET_COL override: {TARGET_COL}")

Results root: /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct
Model ID: xgboost
Run: full_trainval_12ep_1seed_MI_correct
TARGET_COL override: None


## 2. Load Run Manifest and Final-Model Artifacts
**Purpose:** Reconstruct the exported run context from the manifest and load the final fitted model required for analysis.<br>
**Inputs:** `MODEL_ID`, `RUN_NAME`, optional `TARGET_COL`, and the manifest-linked artifact files produced by training.<br>
**Outputs:** `run_ctx`, resolved manifest metadata, loaded final model/scaler artifacts, and the modelling dataframe with OOF columns.<br>
**How to Verify:** confirm the printed manifest path, model-data path, model path, exported-model label, and any selected-variant metadata before generating plots.


In [3]:
# `load_run_context` requires the run manifest and the exported model-data-with-OOF table.
# Optional metrics/tuning summaries are loaded only when the manifest points to existing files.
run_ctx = load_run_context(MODEL_ID, RUN_NAME, TARGET_COL)
manifest_path = run_ctx.manifest_path
manifest = run_ctx.manifest
exported_model_info = get_exported_model_info(manifest)
exported_model_label = format_exported_model_label(exported_model_info)
selection_metric_value = exported_model_info["selection_metric_value"]
selection_metric_value_display = (
    f"{selection_metric_value:.6f}"
    if isinstance(selection_metric_value, (int, float))
    and not isinstance(selection_metric_value, bool)
    else str(selection_metric_value)
)

if manifest["model_id"] not in {"xgboost", "gam"}:
    raise NotImplementedError(
        f"Model inference analysis is not implemented yet for model_id={manifest['model_id']}"
    )

target_col = run_ctx.target_col
feature_cols = run_ctx.feature_cols
TABLES_DIR = run_ctx.tables_dir
PLOTS_DIR = run_ctx.plots_dir
nested_resampling = run_ctx.nested_resampling
final_model = run_ctx.final_model
model_data_path = run_ctx.model_data_path
model_path = Path(final_model["model_path"])
full_data_tuning_summary_path = run_ctx.full_data_tuning_summary_path
full_data_tuning_summary = run_ctx.full_data_tuning_summary or {}

model_df_oof = run_ctx.model_df_oof
X = model_df_oof[feature_cols]
discrete_feature_names = frozenset(
    col for col in feature_cols if col in MODEL_SETTING_COLS
)
selected_variant_name = exported_model_info["name"]
selected_target_mode = exported_model_info["target_mode"]
winning_variant_model_id = final_model.get(
    "selected_variant_model_id", manifest.get("winning_variant_model_id")
)
winning_variant_manifest_path = final_model.get("selected_variant_manifest_path")
selected_variant_nested_summary = final_model.get(
    "selected_variant_nested_summary",
    full_data_tuning_summary.get("selected_variant_nested_summary", {}),
)
full_data_best_cv_rmse = final_model.get(
    "selected_full_data_best_cv_rmse",
    full_data_tuning_summary.get("full_data_best_cv_rmse"),
)

# Model loading is model-family specific, but both branches must produce `X_for_model`
# aligned with the feature columns stored in the manifest.
if MODEL_ID == "xgboost":
    model = xgb.XGBRegressor()
    model.load_model(model_path)
    X_for_model = X
elif MODEL_ID == "gam":
    scaler_path = Path(final_model["scaler_path"])
    with model_path.open("rb") as f:
        model = pickle.load(f)
    with scaler_path.open("rb") as f:
        scaler = pickle.load(f)
    X_for_model = scaler.transform(X)
else:
    raise NotImplementedError(
        f"Model inference analysis is not implemented yet for model_id={MODEL_ID}"
    )

print(f"Loaded manifest: {manifest_path}")
print(f"Loaded model data: {model_data_path}")
print(f"Loaded final model: {model_path}")
print(f"Target: {target_col} | Features: {len(feature_cols)}")
print(f"Exported model: {exported_model_label}")
print(
    "Nested selection metric: "
    f"{exported_model_info['selection_metric_name']}={selection_metric_value_display}"
)
if winning_variant_model_id is not None:
    print(f"Nested-CV winning variant: {winning_variant_model_id}")
if winning_variant_manifest_path is not None:
    print(f"Winning variant manifest: {winning_variant_manifest_path}")
if selected_variant_nested_summary:
    print("Winning variant nested summary:")
    print(json.dumps(selected_variant_nested_summary, indent=2))
if full_data_best_cv_rmse is not None:
    print(f"Full-data retuning best_cv_rmse: {full_data_best_cv_rmse:.6f}")
print("Final-model tuning summary:")
print(json.dumps(full_data_tuning_summary, indent=2))

Loaded manifest: /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/tables/run_manifest_ml_ade_log.json
Loaded model data: /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/tables/model_data_with_oof_ml_ade_log.csv
Loaded final model: /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/xgb_model_ml_ade_log.json
Target: ml_ade_log | Features: 4
Exported model: XGBoost (xgboost, target_mode=log)
Nested selection metric: best_cv_score=0.175739
Final-model tuning summary:
{
  "model_id": "xgboost",
  "run_name": "full_trainval_12ep_1seed_MI_correct",
  "target_col": "ml_ade_log",
  "tuning_metric_name": "rmse",
  "best_params": {
    "learning_rate": 0.017398074711291726,
    "max_depth": 10,
    "min_child_weight": 16,
    "subsample": 0.969749470

## 3. Feature Effect Exports and Global Ranking
**Purpose:** Quantify which features matter most to the final exported model and export one unified per-row feature-effect table for downstream clustering.<br>
**Inputs:** loaded model, feature matrix, and model-specific explanation logic.<br>
**Outputs:** `feature_effect_importance_<target>.csv`, `feature_effects_<target>.csv`, and a saved importance plot.<br>
**How to Verify:** the ranking table should contain one row per feature, the feature-effect table should contain one `effect__<feature>` column per feature plus `effect_base_value`, and the saved plot path should be printed in the final section.


In [4]:
# Use a model-specific explanation method while keeping the exported feature-effect
# table/ranking schema common across XGBoost and GAM runs.
feature_effect_table_path = TABLES_DIR / f"feature_effects_{target_col}.csv"
importance_table_path = TABLES_DIR / f"feature_effect_importance_{target_col}.csv"
importance_plot_path = PLOTS_DIR / f"feature_effect_importance_{target_col}.png"

if MODEL_ID == "xgboost":
    try:
        explainer = shap.Explainer(model, X_for_model)
        shap_exp = explainer(X_for_model)
    except NotImplementedError as exc:
        if "Categorical split is not yet supported" in str(exc):
            print(
                "SHAP fallback: using TreeExplainer with "
                "feature_perturbation='tree_path_dependent' for categorical splits."
            )
            explainer = shap.TreeExplainer(
                model, feature_perturbation="tree_path_dependent"
            )
            shap_exp = explainer(X_for_model)
        else:
            raise

    feature_effect_values = shap_exp.values

    importance_df = build_feature_effect_importance_table(
        model_id=MODEL_ID,
        feature_cols=feature_cols,
        effect_values=feature_effect_values,
    )
    display(importance_df)
    importance_df.to_csv(importance_table_path, index=False)

    feature_effect_df = prepare_feature_effect_export(
        model_df_oof=model_df_oof,
        feature_cols=feature_cols,
        effect_values=feature_effect_values,
        base_values=getattr(shap_exp, "base_values", None),
    )
    feature_effect_df.to_csv(feature_effect_table_path, index=False)

    ranked_features = importance_df["feature"].tolist()
    effect_features = ranked_features[:16]

    plt.figure(figsize=(10, 6))
    shap.summary_plot(feature_effect_values, X_for_model, show=False, max_display=20)
    plt.title(f"SHAP beeswarm — {exported_model_label}")
    beeswarm_path = PLOTS_DIR / f"shap_beeswarm_{target_col}.png"
    plt.tight_layout()
    plt.savefig(beeswarm_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

    fig, ax = plt.subplots(figsize=(8, max(4, len(feature_cols) * 0.4)))
    plot_df = importance_df.sort_values("mean_abs_shap", ascending=True)
    ax.barh(plot_df["feature"], plot_df["mean_abs_shap"], color="steelblue")
    ax.set_xlabel("Mean |SHAP| value")
    ax.set_title(f"SHAP importance — {exported_model_label}")
    plt.tight_layout()
    plt.savefig(importance_plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"Feature-effect ranking table saved to: {importance_table_path}")
    print(f"Feature-effect table saved to:         {feature_effect_table_path}")
    print(f"SHAP beeswarm saved to:               {beeswarm_path}")
    print(f"Feature-effect importance plot saved to: {importance_plot_path}")
elif MODEL_ID == "gam":
    p_values = np.asarray(
        model.statistics_["p_values"][: len(feature_cols)], dtype=float
    )
    feature_effect_values, effect_base_values = compute_gam_feature_effects(
        model=model,
        X_scaled=X_for_model,
        feature_cols=feature_cols,
    )

    feature_effect_df = prepare_feature_effect_export(
        model_df_oof=model_df_oof,
        feature_cols=feature_cols,
        effect_values=feature_effect_values,
        base_values=effect_base_values,
    )
    feature_effect_df.to_csv(feature_effect_table_path, index=False)

    importance_df = build_feature_effect_importance_table(
        model_id=MODEL_ID,
        feature_cols=feature_cols,
        p_values=p_values,
    )
    display(importance_df)
    importance_df.to_csv(importance_table_path, index=False)

    ranked_features = importance_df["feature"].tolist()
    effect_features = ranked_features[:16]

    fig, ax = plt.subplots(figsize=(8, max(4, len(feature_cols) * 0.4)))
    plot_df = importance_df.sort_values("neg_log10_p_value", ascending=True)
    ax.barh(plot_df["feature"], plot_df["neg_log10_p_value"], color="steelblue")
    ax.axvline(
        x=-np.log10(0.05), color="red", linestyle="--", label="p=0.05", alpha=0.7
    )
    ax.axvline(
        x=-np.log10(0.01), color="orange", linestyle="--", label="p=0.01", alpha=0.7
    )
    ax.set_xlabel("-log10(p-value)")
    ax.set_title(f"Smooth effect significance — {exported_model_label}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(importance_plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    beeswarm_path = None
    print(f"Feature-effect ranking table saved to: {importance_table_path}")
    print(f"Feature-effect table saved to:         {feature_effect_table_path}")
    print(f"Feature-effect importance plot saved to: {importance_plot_path}")
else:
    raise NotImplementedError(
        f"Model inference analysis is not implemented yet for model_id={MODEL_ID}"
    )

print("Features selected for downstream effect plots (up to 16, ranked):")
print(effect_features)

  0%|                   | 104/26886 [00:11<47:12]       

  0%|                   | 113/26886 [00:12<47:23]       

  0%|                   | 125/26886 [00:13<46:23]       

  0%|                   | 132/26886 [00:14<47:17]       

  1%|                   | 141/26886 [00:15<47:25]       

...

100%|===================| 26846/26886 [31:04<00:02]       

100%|===================| 26862/26886 [31:05<00:01]       

100%|===================| 26878/26886 [31:06<00:00]       

,feature,global_rank,mean_abs_shap,importance_metric,importance_value,importance_ascending
0,std_speed,1,0.108838,mean_abs_shap,0.108838,False
1,max_speed,2,0.103074,mean_abs_shap,0.103074,False
2,heading_change_per_sec,3,0.089916,mean_abs_shap,0.089916,False
3,mean_acceleration,4,0.025765,mean_abs_shap,0.025765,False


Feature-effect ranking table saved to: /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/tables/feature_effect_importance_ml_ade_log.csv
Feature-effect table saved to:         /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/tables/feature_effects_ml_ade_log.csv
SHAP beeswarm saved to:               /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/plots/shap_beeswarm_ml_ade_log.png
Feature-effect importance plot saved to: /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/plots/feature_effect_importance_ml_ade_log.png
Features selected for downstream effect plots (up to 16, ranked):
['std_speed', 'max_speed', 'heading_change_per_sec', 'mean_acceleration']


## 4. Feature Effect Plots
**Purpose:** Visualize how the fitted model responds to changes in individual features.<br>
**Inputs:** loaded model, modelling feature matrix, and the selected effect-feature subset.<br>
**Outputs:** a grid of model-specific feature-effect plots saved to disk.<br>
**How to Verify:** confirm the grid renders without missing-feature errors and that each plotted feature exists in `feature_cols`.


In [5]:
# Plot feature effects using the fitted final model rather than OOF predictions so the
# shapes reflect the exact exported artifact that downstream users will inspect.
# Discrete model settings (attention_radius_m, history_sec, prediction_sec) are rendered
# as bar charts at their actual levels; continuous features keep the smooth line/PDP.
fig, axes = plt.subplots(4, 4, figsize=(22, 18))
axes = axes.flatten()

display_target_col = resolve_raw_metric_col(manifest, target_col)
target_mode_label = "log" if selected_target_mode == "log" else "raw"
target_label = f"{display_target_col} ({target_mode_label})"

if MODEL_ID == "xgboost":
    # Pre-compute raw PDP values once; reused for both the centered additive and
    # multiplicative plots below.
    _pdp_cache = {}
    for feat in effect_features:
        if feat in discrete_feature_names:
            unique_vals = sorted(X[feat].unique())
            raw_means = []
            for v in unique_vals:
                X_copy = X_for_model.copy()
                X_copy[feat] = v
                raw_means.append(float(model.predict(X_copy).mean()))
            y = np.array(raw_means)
            _pdp_cache[feat] = {
                "type": "discrete",
                "x": unique_vals,
                "y_centered": y - y.mean(),
            }
        else:
            res = partial_dependence(
                model, X_for_model, features=[feat], kind="average", grid_resolution=40
            )
            y = res["average"][0]
            _pdp_cache[feat] = {
                "type": "continuous",
                "x": res["grid_values"][0],
                "y_centered": y - y.mean(),
            }

    for ax, feat in zip(axes, effect_features):
        d = _pdp_cache[feat]
        y_c = d["y_centered"]
        if d["type"] == "discrete":
            x_pos = range(len(d["x"]))
            ax.bar(
                x_pos,
                y_c,
                color="steelblue",
                alpha=0.8,
                edgecolor="black",
                linewidth=0.8,
            )
            ax.axhline(0, color="red", linestyle="--", alpha=0.5)
            ax.set_xticks(list(x_pos))
            ax.set_xticklabels([f"{v:.4g}" for v in d["x"]], rotation=30, ha="right")
            ax.set_title(f"PDP (discrete) — {feat}")
        else:
            ax.plot(d["x"], y_c, color="steelblue", linewidth=2)
            ax.axhline(0, color="red", linestyle="--", alpha=0.5)
            ax.set_title(f"PDP — {feat}")
        ax.set_xlabel(feat)
        ax.set_ylabel(f"Centered PDP — log({display_target_col})")

elif MODEL_ID == "gam":
    ylabel = (
        f"Additive effect on {display_target_col} (log scale)"
        if selected_target_mode == "log"
        else f"Additive effect on {display_target_col}"
    )
    for ax, feat in zip(axes, effect_features):
        feat_idx = feature_cols.index(feat)
        if feat in discrete_feature_names:
            # Factor term: evaluate at the actual discrete levels observed in the data
            unique_raw_vals = np.array(sorted(X[feat].unique()))
            x_scaled = (unique_raw_vals - scaler.mean_[feat_idx]) / scaler.scale_[
                feat_idx
            ]
            XX = np.tile(X_for_model.mean(axis=0), (len(x_scaled), 1))
            XX[:, feat_idx] = x_scaled
            pdep, confi = model.partial_dependence(term=feat_idx, X=XX, width=0.95)
            x_pos = np.arange(len(unique_raw_vals))
            ax.bar(
                x_pos,
                pdep,
                color="steelblue",
                alpha=0.8,
                edgecolor="black",
                linewidth=0.8,
            )
            ax.errorbar(
                x_pos,
                pdep,
                yerr=[pdep - confi[:, 0], confi[:, 1] - pdep],
                fmt="none",
                color="black",
                capsize=4,
                linewidth=1.2,
            )
            ax.axhline(0, color="red", linestyle="--", alpha=0.5)
            ax.set_xticks(x_pos)
            ax.set_xticklabels(
                [f"{v:.4g}" for v in unique_raw_vals], rotation=30, ha="right"
            )
            ax.set_title(f"Factor effect — {feat}")
        else:
            XX = model.generate_X_grid(term=feat_idx)
            pdep, confi = model.partial_dependence(term=feat_idx, X=XX, width=0.95)
            x_scaled_cont = XX[:, feat_idx]
            x_original = (
                x_scaled_cont * scaler.scale_[feat_idx] + scaler.mean_[feat_idx]
            )
            ax.plot(x_original, pdep, color="steelblue", linewidth=2)
            ax.fill_between(
                x_original, confi[:, 0], confi[:, 1], alpha=0.2, color="steelblue"
            )
            ax.axhline(0, color="red", linestyle="--", alpha=0.5)
            ax.set_title(f"Smooth additive effect — {feat}")
        ax.set_xlabel(feat)
        ax.set_ylabel(ylabel)
else:
    raise NotImplementedError(
        f"Model inference analysis is not implemented yet for model_id={MODEL_ID}"
    )

for ax in axes[len(effect_features) :]:
    ax.set_visible(False)

plt.suptitle(
    f"Feature effect plots for {exported_model_label} — Target: {target_label}",
    fontsize=16,
    y=1.02,
)
plt.tight_layout()
effect_plot_path = PLOTS_DIR / f"feature_effects_grid_{target_col}.png"
plt.savefig(effect_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Feature effect grid saved for the ranked feature set.")
print(f"Feature effect grid path: {effect_plot_path}")
if MODEL_ID == "gam" and selected_target_mode == "log":
    print(
        "GAM note: additive effects are exported and plotted on the log/link scale; they imply multiplicative changes on the original target scale."
    )

# Multiplicative-scale PDP: exp(centered PDP) gives a factor relative to the geometric
# mean prediction, i.e. how much higher/lower ml_ade is at each feature value compared
# to the dataset average. Only meaningful when the model fits a log-transformed target.
effect_plot_mult_path = None
if MODEL_ID == "xgboost" and target_col.endswith("_log"):
    fig_mult, axes_mult = plt.subplots(4, 4, figsize=(22, 18))
    axes_mult = axes_mult.flatten()

    for ax, feat in zip(axes_mult, effect_features):
        d = _pdp_cache[feat]
        y_mult = np.exp(d["y_centered"])
        if d["type"] == "discrete":
            x_pos = range(len(d["x"]))
            ax.bar(
                x_pos,
                y_mult,
                color="darkorange",
                alpha=0.8,
                edgecolor="black",
                linewidth=0.8,
            )
            ax.axhline(1.0, color="red", linestyle="--", alpha=0.5)
            ax.set_xticks(list(x_pos))
            ax.set_xticklabels([f"{v:.4g}" for v in d["x"]], rotation=30, ha="right")
            ax.set_title(f"PDP (discrete) — {feat}")
        else:
            ax.plot(d["x"], y_mult, color="darkorange", linewidth=2)
            ax.axhline(1.0, color="red", linestyle="--", alpha=0.5)
            ax.set_title(f"PDP — {feat}")
        ax.set_xlabel(feat)
        ax.set_ylabel(
            f"Multiplicative effect on {display_target_col} (relative to dataset mean)"
        )

    for ax in axes_mult[len(effect_features) :]:
        ax.set_visible(False)

    plt.suptitle(
        f"PDP multiplicative effects for {exported_model_label} — Target: {display_target_col}",
        fontsize=16,
        y=1.02,
    )
    plt.tight_layout()
    effect_plot_mult_path = PLOTS_DIR / f"feature_effects_grid_mult_{target_col}.png"
    plt.savefig(effect_plot_mult_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Feature effect multiplicative grid path: {effect_plot_mult_path}")

Feature effect grid saved for the ranked feature set.
Feature effect grid path: /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/plots/feature_effects_grid_ml_ade_log.png


Feature effect multiplicative grid path: /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/plots/feature_effects_grid_mult_ml_ade_log.png


## 5. Cohort Violin Plots by Actual Performance
**Purpose:** Compare feature distributions between high-performing and low-performing trajectories using the actual target on the original scale.<br>
**Inputs:** `target_orig`, configured quantile threshold, and selected cohort features.<br>
**Outputs:** quantile-defined cohort masks and a saved violin-plot figure.<br>
**How to Verify:** confirm the printed thresholds are sensible for the target distribution and that both cohorts contain non-zero rows.


In [6]:
# Cohorts are defined on the actual target, not the prediction, so the comparison stays
# anchored to observed performance instead of (interpretable)model self-assessment.
actual_metric_col = manifest.get(
    "raw_target_col", target_col[:-4] if target_col.endswith("_log") else target_col
)
actual_metric = model_df_oof["target_orig"].to_numpy()
poor_well_quantile = manifest.get("analysis", {}).get("poor_well_quantile", 0.20)

if LOWER_IS_BETTER:
    best_threshold = np.quantile(actual_metric, poor_well_quantile)
    worst_threshold = np.quantile(actual_metric, 1.0 - poor_well_quantile)
    best_mask = actual_metric <= best_threshold
    worst_mask = actual_metric >= worst_threshold
    best_label = f"Best (bottom {poor_well_quantile:.0%})"
    worst_label = f"Worst (top {poor_well_quantile:.0%})"
else:
    best_threshold = np.quantile(actual_metric, 1.0 - poor_well_quantile)
    worst_threshold = np.quantile(actual_metric, poor_well_quantile)
    best_mask = actual_metric >= best_threshold
    worst_mask = actual_metric <= worst_threshold
    best_label = f"Best (top {poor_well_quantile:.0%})"
    worst_label = f"Worst (bottom {poor_well_quantile:.0%})"

top_violin_features = ranked_features[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = np.array(axes).reshape(-1)

for ax, feat in zip(axes, top_violin_features):
    long_df = pd.concat(
        [
            pd.DataFrame(
                {"cohort": best_label, "value": X.loc[best_mask, feat].values}
            ),
            pd.DataFrame(
                {"cohort": worst_label, "value": X.loc[worst_mask, feat].values}
            ),
        ],
        ignore_index=True,
    )

    sns.violinplot(data=long_df, x="cohort", y="value", ax=ax, inner="box", cut=0)
    ax.set_title(feat)
    ax.set_xlabel("")
    ax.set_ylabel(feat)
    ax.tick_params(axis="x", rotation=12)

for ax in axes[len(top_violin_features) :]:
    ax.set_visible(False)

plt.suptitle(
    f"Top 6 feature distributions of {exported_model_label} by actual {actual_metric_col} performance cohort",
    fontsize=16,
    y=1.02,
)
plt.tight_layout()
cohort_plot_path = (
    PLOTS_DIR / f"cohort_violin_top_features_actual_{actual_metric_col}.png"
)
plt.savefig(cohort_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Actual metric used for cohorting: {actual_metric_col}")
print(f"Best cohort size: {best_mask.sum()}")
print(f"Worst cohort size: {worst_mask.sum()}")
print(f"Best threshold: {best_threshold:.6f}")
print(f"Worst threshold: {worst_threshold:.6f}")
print(f"Cohort violin plot path: {cohort_plot_path}")

Actual metric used for cohorting: ml_ade
Best cohort size: 5378
Worst cohort size: 5378
Best threshold: 0.117489
Worst threshold: 0.740227
Cohort violin plot path: /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/plots/cohort_violin_top_features_actual_ml_ade.png


## 6. Saved Artifacts
**Purpose:** Summarize every artifact produced or consumed by this analysis run.<br>
**Inputs:** saved table/plot paths and the resolved run metadata.<br>
**Outputs:** a compact audit trail of manifest, model, tuning summary, and generated analysis artifacts.<br>
**How to Verify:** confirm each printed path exists and refers to the same run id selected at the top of the notebook.


In [7]:
print("Saved artifacts:")
print(f"- Run manifest:                 {manifest_path}")
print(f"- Final model:                  {model_path}")
print(f"- Tuning summary:               {full_data_tuning_summary_path}")
print(f"- Feature-effect ranking table: {importance_table_path}")
print(f"- Feature-effect table:         {feature_effect_table_path}")
print(f"- Feature-effect plot:          {importance_plot_path}")
print(f"- Effect plot grid:             {effect_plot_path}")
if effect_plot_mult_path is not None:
    print(f"- Effect plot grid (mult):      {effect_plot_mult_path}")
print(f"- Cohort violin plot:           {cohort_plot_path}")
print(f"- Winning variant ID:           {winning_variant_model_id}")
if winning_variant_manifest_path is not None:
    print(f"- Winning variant manifest:     {winning_variant_manifest_path}")
if MODEL_ID == "xgboost":
    print(f"- SHAP beeswarm:                {beeswarm_path}")
if MODEL_ID == "gam":
    print(f"- Scaler:                       {scaler_path}")
print(f"- Plot directory:               {PLOTS_DIR}")

Saved artifacts:
- Run manifest:                 /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/tables/run_manifest_ml_ade_log.json
- Final model:                  /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/xgb_model_ml_ade_log.json
- Tuning summary:               /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/tables/full_data_tuning_optuna_summary_ml_ade_log.json
- Feature-effect ranking table: /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_correct/tables/feature_effect_importance_ml_ade_log.csv
- Feature-effect table:         /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/full_trainval_12ep_1seed_MI_c